In [7]:
import pandas as pd
import numpy as np
import sys
sys.path.append(r"D:\HopeAI\Assignments\MyModules")

from data_analysis_utils import Preprocessing

stocks_df = pd.read_csv(r"D:\HopeAI\Assignments\ML and DS Capstone\1. Data Collection\nifty50_stocks.csv")
nifty_df = pd.read_csv(r'D:\HopeAI\Assignments\ML and DS Capstone\1. Data Collection\NIFTY50.csv')
stocks_df


,Date,Close,High,Low,Open,Volume,Stock,Sector,Industry,Market Cap
0,2014-01-01,37.932781,38.095098,37.022394,37.234111,7564701,ADANIENT,Energy,Thermal Coal,2994291081216
1,2014-01-02,36.041435,37.932782,35.752089,37.262341,17188171,ADANIENT,Energy,Thermal Coal,2994291081216
2,2014-01-03,35.053421,36.140239,34.510010,35.709744,11525782,ADANIENT,Energy,Thermal Coal,2994291081216
3,2014-01-06,35.575649,35.977915,34.538232,34.989899,10660990,ADANIENT,Energy,Thermal Coal,2994291081216
4,2014-01-07,34.150085,35.977917,34.015998,35.759142,11002957,ADANIENT,Energy,Thermal Coal,2994291081216
...,...,...,...,...,...,...,...,...,...,...
123109,2024-12-23,296.077271,298.721676,293.288630,294.827200,6693583,WIPRO,Technology,Information Technology Services,2532097654784
123110,2024-12-24,293.577087,297.327359,291.173082,295.692624,8837902,WIPRO,Technology,Information Technology Services,2532097654784
123111,2024-12-26,293.336670,295.163737,292.327000,293.673236,6516148,WIPRO,Technology,Information Technology Services,2532097654784
123112,2024-12-27,297.231201,298.962073,293.000158,293.480959,8063921,WIPRO,Technology,Information Technology Services,2532097654784


In [8]:

stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])
stocks_df['Year'] = stocks_df['Date'].dt.year

nifty_df['Date'] = pd.to_datetime(nifty_df['Date'])
nifty_df.set_index('Date', inplace=True)
nifty_returns = nifty_df['Close'].pct_change().dropna()
nifty_returns

feature_rows = []
stocks = stocks_df['Stock'].unique()
stocks

for stock in stocks:
    stock_data = stocks_df[stocks_df['Stock'] == stock].copy()
    
    for year, year_data in stock_data.groupby('Year'):
        daily_returns = year_data['Close'].pct_change().dropna()
        monthly_prices = year_data.set_index('Date')['Close'].resample('ME').ffill()
        monthly_returns = monthly_prices.pct_change().dropna()
        

        # Collected features
        sector = year_data['Sector'].unique().item()
        industry = year_data['Industry'].unique().item()
        market_cap = year_data['Market Cap'].unique().item()

        # Return features
        total_annual_return = (year_data['Close'].iloc[-1] / year_data['Close'].iloc[0]) - 1
        avg_monthly_return = monthly_returns.mean()
        std_monthly_return = monthly_returns.std()
        
        
        # Volatility / Price Range
        max_drawdown = ((year_data['Close'].cummax() - year_data['Close']) / year_data['Close'].cummax()).max()
        std_daily_return = daily_returns.std()
        pos_days_pct = (daily_returns > 0).mean()
        neg_days_pct = (daily_returns < 0).mean()
        avg_daily_TR = (year_data['High'] - year_data['Low']).mean()
        max_daily_TR = (year_data['High'] - year_data['Low']).max()
        
        # Volume
        avg_volume = year_data['Volume'].mean()
        volume_spike_pct = (year_data['Volume'] > 2 * avg_volume).mean()
        
        # Trend / Momentum
        twelve_month_momentum = (monthly_prices.iloc[-1] - monthly_prices.iloc[0]) / monthly_prices.iloc[0]
        months_pos_pct = (monthly_returns > 0).mean()
        
        # Market relationship
        year_data = year_data.set_index('Date').sort_index()
        daily_returns = year_data['Close'].pct_change().dropna()
        nifty_year = nifty_returns.loc[daily_returns.index.min(): daily_returns.index.max()]

        combined = pd.DataFrame({
            'Stock_Return': daily_returns,
            'NIFTY_Return': nifty_year
        }).dropna()

        if len(combined) > 30:  # require at least 30 days
            cov_matrix = np.cov(combined['Stock_Return'], combined['NIFTY_Return'])
            beta = cov_matrix[0,1] / cov_matrix[1,1] if cov_matrix[1,1] != 0 else np.nan
            corr_nifty = combined['Stock_Return'].corr(combined['NIFTY_Return'])
        else:
            beta = np.nan
            corr_nifty = np.nan

        feature_row = {
            'Stock': stock,
            'Sector': sector,
            'Industry': industry,
            'Market Cap': market_cap,
            'Year': year,
            'Total_Annual_Return': total_annual_return*100,  # Percentage
            'Avg_Monthly_Return': avg_monthly_return*100,  # Percentage
            'Std_Monthly_Return': std_monthly_return*100,  # Percentage
            'Max_Drawdown': max_drawdown*100,  # Percentage
            'Std_Daily_Return': std_daily_return*100,  # Percentage
            'Pos_Days_Pct': pos_days_pct*100,  # Percentage
            'Neg_Days_Pct': neg_days_pct*100,  # Percentage
            'Avg_Daily_TR': avg_daily_TR,
            'Max_Daily_TR': max_daily_TR,
            'Avg_Volume': avg_volume,
            'Volume_Spike_Pct': volume_spike_pct*100,  # Percentage
            '12M_Momentum': twelve_month_momentum*100,  # Percentage
            'Months_Pos_Pct': months_pos_pct*100,  # Percentage
            'Beta_vs_NIFTY': beta,
            'Corr_with_NIFTY': corr_nifty
        }
        
        feature_rows.append(feature_row)

# Create DataFrame
features_df = pd.DataFrame(feature_rows)

# Save feature matrix
features_df.to_csv('stock_risk_features.csv', index=False)
features_df



,Stock,Sector,Industry,Market Cap,Year,Total_Annual_Return,Avg_Monthly_Return,Std_Monthly_Return,Max_Drawdown,Std_Daily_Return,Pos_Days_Pct,Neg_Days_Pct,Avg_Daily_TR,Max_Daily_TR,Avg_Volume,Volume_Spike_Pct,12M_Momentum,Months_Pos_Pct,Beta_vs_NIFTY,Corr_with_NIFTY
0,ADANIENT,Energy,Thermal Coal,2994291081216,2014,84.785613,7.637515,14.186042,22.867706,3.010790,52.674897,46.090535,2.352871,13.486429,1.861417e+07,8.196721,107.917632,63.636364,1.598832,0.426155
1,ADANIENT,Energy,Thermal Coal,2994291081216,2015,-37.685498,-3.887377,19.560876,68.997003,4.156785,45.714286,52.653061,3.742656,239.273886,1.120364e+07,7.317073,-51.324566,54.545455,1.850348,0.457929
2,ADANIENT,Energy,Thermal Coal,2994291081216,2016,-14.119856,1.611336,15.492383,34.794663,2.639648,53.469388,45.306122,1.581010,4.432787,6.074467e+06,6.097561,6.215522,54.545455,1.649825,0.597262
3,ADANIENT,Energy,Thermal Coal,2994291081216,2017,117.329561,6.434280,8.526439,28.780005,3.281672,56.680162,42.510121,2.889061,19.576553,1.295658e+07,8.064516,91.893377,81.818182,1.912510,0.332076
4,ADANIENT,Energy,Thermal Coal,2994291081216,2018,78.500944,5.578514,26.632411,43.633455,3.637141,54.693878,45.306122,6.293391,30.900989,1.475299e+07,11.382114,39.024769,36.363636,2.142941,0.478182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,WIPRO,Technology,Information Technology Services,2532097654784,2020,56.564122,5.078310,11.196238,36.623083,2.500823,52.000000,47.200000,3.742075,16.643016,2.239159e+07,9.163347,63.112320,63.636364,0.670407,0.529060
498,WIPRO,Technology,Information Technology Services,2532097654784,2021,84.735387,5.200053,6.752821,12.734524,1.763293,57.085020,42.914980,6.545712,21.253327,2.143325e+07,10.080645,71.177270,72.727273,0.849918,0.476230
499,WIPRO,Technology,Information Technology Services,2532097654784,2022,-44.808074,-3.105960,6.336069,47.492937,1.706648,47.368421,52.631579,4.864650,17.585685,1.479087e+07,3.629032,-30.838344,27.272727,1.083736,0.691712
500,WIPRO,Technology,Information Technology Services,2532097654784,2023,20.203754,1.701905,6.272716,14.238744,1.237919,48.360656,51.229508,3.161690,15.496817,1.015820e+07,6.938776,18.164716,54.545455,1.108324,0.554158


In [9]:
features_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Stock                502 non-null    object 
 1   Sector               502 non-null    object 
 2   Industry             502 non-null    object 
 3   Market Cap           502 non-null    int64  
 4   Year                 502 non-null    int64  
 5   Total_Annual_Return  502 non-null    float64
 6   Avg_Monthly_Return   502 non-null    float64
 7   Std_Monthly_Return   501 non-null    float64
 8   Max_Drawdown         502 non-null    float64
 9   Std_Daily_Return     502 non-null    float64
 10  Pos_Days_Pct         502 non-null    float64
 11  Neg_Days_Pct         502 non-null    float64
 12  Avg_Daily_TR         502 non-null    float64
 13  Max_Daily_TR         502 non-null    float64
 14  Avg_Volume           502 non-null    float64
 15  Volume_Spike_Pct     502 non-null    flo

In [10]:
print(features_df.isnull().mean()*100)


Stock                  0.000000
Sector                 0.000000
Industry               0.000000
Market Cap             0.000000
Year                   0.000000
Total_Annual_Return    0.000000
Avg_Monthly_Return     0.000000
Std_Monthly_Return     0.199203
Max_Drawdown           0.000000
Std_Daily_Return       0.000000
Pos_Days_Pct           0.000000
Neg_Days_Pct           0.000000
Avg_Daily_TR           0.000000
Max_Daily_TR           0.000000
Avg_Volume             0.000000
Volume_Spike_Pct       0.000000
12M_Momentum           0.000000
Months_Pos_Pct         0.000000
Beta_vs_NIFTY          0.199203
Corr_with_NIFTY        0.199203
dtype: float64


In [11]:
quan, qual = Preprocessing.quanQual(features_df)

print(quan, qual)


Index(['Market Cap', 'Year', 'Total_Annual_Return', 'Avg_Monthly_Return',
       'Std_Monthly_Return', 'Max_Drawdown', 'Std_Daily_Return',
       'Pos_Days_Pct', 'Neg_Days_Pct', 'Avg_Daily_TR', 'Max_Daily_TR',
       'Avg_Volume', 'Volume_Spike_Pct', '12M_Momentum', 'Months_Pos_Pct',
       'Beta_vs_NIFTY', 'Corr_with_NIFTY'],
      dtype='object') Index(['Stock', 'Sector', 'Industry'], dtype='object')


### Missing Values


In [12]:
dataset = Preprocessing.simple_missing(features_df, quan, qual) 
dataset


,Stock,Sector,Industry,Market Cap,Year,Total_Annual_Return,Avg_Monthly_Return,Std_Monthly_Return,Max_Drawdown,Std_Daily_Return,Pos_Days_Pct,Neg_Days_Pct,Avg_Daily_TR,Max_Daily_TR,Avg_Volume,Volume_Spike_Pct,12M_Momentum,Months_Pos_Pct,Beta_vs_NIFTY,Corr_with_NIFTY
0,ADANIENT,Energy,Thermal Coal,2994291081216,2014,84.785613,7.637515,14.186042,22.867706,3.010790,52.674897,46.090535,2.352871,13.486429,1.861417e+07,8.196721,107.917632,63.636364,1.598832,0.426155
1,ADANIENT,Energy,Thermal Coal,2994291081216,2015,-37.685498,-3.887377,19.560876,68.997003,4.156785,45.714286,52.653061,3.742656,239.273886,1.120364e+07,7.317073,-51.324566,54.545455,1.850348,0.457929
2,ADANIENT,Energy,Thermal Coal,2994291081216,2016,-14.119856,1.611336,15.492383,34.794663,2.639648,53.469388,45.306122,1.581010,4.432787,6.074467e+06,6.097561,6.215522,54.545455,1.649825,0.597262
3,ADANIENT,Energy,Thermal Coal,2994291081216,2017,117.329561,6.434280,8.526439,28.780005,3.281672,56.680162,42.510121,2.889061,19.576553,1.295658e+07,8.064516,91.893377,81.818182,1.912510,0.332076
4,ADANIENT,Energy,Thermal Coal,2994291081216,2018,78.500944,5.578514,26.632411,43.633455,3.637141,54.693878,45.306122,6.293391,30.900989,1.475299e+07,11.382114,39.024769,36.363636,2.142941,0.478182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,WIPRO,Technology,Information Technology Services,2532097654784,2020,56.564122,5.078310,11.196238,36.623083,2.500823,52.000000,47.200000,3.742075,16.643016,2.239159e+07,9.163347,63.112320,63.636364,0.670407,0.529060
498,WIPRO,Technology,Information Technology Services,2532097654784,2021,84.735387,5.200053,6.752821,12.734524,1.763293,57.085020,42.914980,6.545712,21.253327,2.143325e+07,10.080645,71.177270,72.727273,0.849918,0.476230
499,WIPRO,Technology,Information Technology Services,2532097654784,2022,-44.808074,-3.105960,6.336069,47.492937,1.706648,47.368421,52.631579,4.864650,17.585685,1.479087e+07,3.629032,-30.838344,27.272727,1.083736,0.691712
500,WIPRO,Technology,Information Technology Services,2532097654784,2023,20.203754,1.701905,6.272716,14.238744,1.237919,48.360656,51.229508,3.161690,15.496817,1.015820e+07,6.938776,18.164716,54.545455,1.108324,0.554158


In [13]:
dataset['Stock'].value_counts()


Stock
ADANIENT      11
ADANIPORTS    11
APOLLOHOSP    11
ASIANPAINT    11
AXISBANK      11
BAJAJ-AUTO    11
BAJFINANCE    11
BAJAJFINSV    11
BEL           11
BHARTIARTL    11
CIPLA         11
COALINDIA     11
DRREDDY       11
EICHERMOT     11
GRASIM        11
HCLTECH       11
HDFCBANK      11
HINDALCO      11
HINDUNILVR    11
INFY          11
ICICIBANK     11
ITC           11
JSWSTEEL      11
NESTLEIND     11
KOTAKBANK     11
LT            11
M&M           11
NTPC          11
MARUTI        11
POWERGRID     11
ONGC          11
TITAN         11
TATAMOTORS    11
RELIANCE      11
SHRIRAMFIN    11
SUNPHARMA     11
SBIN          11
TCS           11
TATACONSUM    11
ULTRACEMCO    11
TRENT         11
TATASTEEL     11
TECHM         11
WIPRO         11
SBILIFE        8
HDFCLIFE       7
JIOFIN         2
Name: count, dtype: int64

In [14]:
# Find stocks that appear at least 11 times
valid_stocks = dataset['Stock'].value_counts()[dataset['Stock'].value_counts() >= 11].index

# Keep only those rows
dataset = dataset[dataset['Stock'].isin(valid_stocks)]


In [15]:
dataset['Stock'].value_counts()


Stock
ADANIENT      11
ADANIPORTS    11
APOLLOHOSP    11
ASIANPAINT    11
AXISBANK      11
BAJAJ-AUTO    11
BAJFINANCE    11
BAJAJFINSV    11
BEL           11
BHARTIARTL    11
CIPLA         11
COALINDIA     11
DRREDDY       11
EICHERMOT     11
GRASIM        11
HCLTECH       11
HDFCBANK      11
HINDALCO      11
HINDUNILVR    11
ICICIBANK     11
INFY          11
ITC           11
JSWSTEEL      11
KOTAKBANK     11
LT            11
M&M           11
MARUTI        11
NESTLEIND     11
NTPC          11
ONGC          11
POWERGRID     11
RELIANCE      11
SHRIRAMFIN    11
SBIN          11
SUNPHARMA     11
TCS           11
TATACONSUM    11
TATAMOTORS    11
TATASTEEL     11
TECHM         11
TITAN         11
TRENT         11
ULTRACEMCO    11
WIPRO         11
Name: count, dtype: int64

In [16]:
import pickle
dataset.to_csv('PreStockFeatures.csv', index=False)
filename='pre_dataset.pkl'
pickle.dump(dataset, open(filename,'wb'))
